# Data Preprocessing Pipeline

**Scope:** Risk of Recidivism assessment only  
**Target:** RawScore

## 0) Environment and Imports

In [ ]:
import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

DATA_PATH = '../data/compas-scores-raw.csv'
OUTPUT_PATH = '../data/compas-scores-processed.csv'
TARGET = 'RawScore'
ASSESSMENT = 'Risk of Recidivism'

## 1) Load Raw Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Raw dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

## 2) Filter to Assessment Scope

We focused on **Risk of Recidivism**.

In [ ]:
n_before = len(df)
df = df[df['DisplayText'] == ASSESSMENT].copy()
print(f"Filtered to '{ASSESSMENT}': {n_before:,} → {len(df):,} rows")
print(f"Unique individuals: {df['Person_ID'].nunique():,}")

## 3) Parse dates and compute age

Convert date strings to datetime objects and compute age at screening (in years).

In [ ]:
df['Screening_Date'] = pd.to_datetime(df['Screening_Date'], format='mixed', dayfirst=False)
df['DateOfBirth'] = pd.to_datetime(df['DateOfBirth'], format='mixed', dayfirst=False)

df['age_at_screening'] = (df['Screening_Date'] - df['DateOfBirth']).dt.days / 365.25

print(f"Age range: {df['age_at_screening'].min():.1f} to {df['age_at_screening'].max():.1f} years")

In [ ]:
n_before = len(df)
df = df[(df['age_at_screening'] >= 10) & (df['age_at_screening'] <= 100)]
print(f"Dropped {n_before - len(df):,} rows with implausible age (<10 or >100)")

## 4) Quality Filtering: Target Variable

Drop rows where the target RawScore is missing.

In [ ]:
n_before = len(df)
df = df.dropna(subset=[TARGET])
print(f"Dropped {n_before - len(df):,} rows with missing RawScore")
print(f"\nTarget statistics:")
print(f"  Mean: {df[TARGET].mean():.2f}")
print(f"  Std:  {df[TARGET].std():.2f}")
print(f"  Min:  {df[TARGET].min():.2f}")
print(f"  Max:  {df[TARGET].max():.2f}")

## 5) Categorical Data Cleaning

**Rationale:** Rare categories create sparse one-hot-encoded columns that, add noise and hurt generalization.

**Decision:** Merge categories representing <1% of population into "Other".

### 5.1 Ethnic_Code_Text Cleanup

In [ ]:
print("Before cleanup:")
print(df['Ethnic_Code_Text'].value_counts(normalize=True) * 100)
print()

In [ ]:
# Fix typo and merge rare categories
df['Ethnic_Code_Text'] = df['Ethnic_Code_Text'].replace({
    'African-Am': 'African-American',  # typo variant
    'Asian': 'Other',                  # <1% representation
    'Native American': 'Other',        # <1% representation
    'Arabic': 'Other',                 # <1% representation
    'Oriental': 'Other',               # <1% representation
})

print("After cleanup:")
print(df['Ethnic_Code_Text'].value_counts(normalize=True) * 100)
print(f"\nCategories reduced from 9 to {df['Ethnic_Code_Text'].nunique()}")

### 5.2 LegalStatus Cleanup

In [ ]:
print("Before cleanup:")
print(df['LegalStatus'].value_counts(normalize=True) * 100)
print()

In [ ]:
# Merge rare categories (<7% representation)
df['LegalStatus'] = df['LegalStatus'].replace({
    'Conditional Release': 'Other',
    'Probation Violator': 'Other',
    'Parole Violator': 'Other',
    'Deferred Sentencing': 'Other',
})

print("After cleanup:")
print(df['LegalStatus'].value_counts(normalize=True) * 100)
print(f"\nCategories reduced from 7 to {df['LegalStatus'].nunique()}")

### 5.3 Drop Zero-Variance Columns

In [ ]:
print("AssessmentReason unique values:", df['AssessmentReason'].nunique())
print(df['AssessmentReason'].value_counts())
print()
print("Language distribution:")
print(df['Language'].value_counts(normalize=True) * 100)

In [ ]:
# AssessmentReason: all identical (zero variance) → drop
# Language: 99% English (near-zero variance) → drop
df = df.drop(columns=['AssessmentReason', 'Language'])

print("Dropped: AssessmentReason (zero variance), Language (99% English)")
print(f"Shape after: {df.shape}")

## 6) Handle Missing Values

Fill categorical and numerical columns with appropriate defaults.

In [ ]:
# Categorical: fill with 'Unknown'
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')

# Numerical: fill with median
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

print(f"Missing values after fill: {df.isnull().sum().sum()}")

## 7) Final Dataset Summary

In [ ]:
print(f"Final processed dataset:")
print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Unique individuals: {df['Person_ID'].nunique():,}")
print(f"\nFeature breakdown:")
print(f"  Categorical: {df.select_dtypes(include=['object']).shape[1]}")
print(f"  Numerical: {df.select_dtypes(include=[np.number]).shape[1]}")
print(f"\nTarget variable (RawScore):")
print(f"  Mean: {df[TARGET].mean():.3f}")
print(f"  Std:  {df[TARGET].std():.3f}")
print(f"  Min:  {df[TARGET].min():.3f}")
print(f"  Max:  {df[TARGET].max():.3f}")

## 8) Save Processed Data

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
print(f"Shape: {df.shape}")